In [ ]:
import yaml
from app.datasets.loader import load_multiple_test_cases, load_test_cases
from app.datasets.validator import validate_dataset_schema
from app.datasets.validator import validate_dataset_schema
from app.client.rag_client import RAGClient
from app.tests.run_tests import run_tests
from app.utils.clean_responses_load_testing import clean_responses_load_testing

In [ ]:
file_name =  'responses_5.4.2'
clean_responses_load_testing(f'./app/data/processed/load/{file_name}', f'{file_name}')

In [ ]:
file_list = [
  './app/data/raw/rapido.xlsx',
]

test_config = {
  'GENERAL_TESTS': True,
  'TIMINGS': {'test': True, 'report': False},
  'TOKENS': {'test': False, 'report': False},
  'FOUNDRYS': {'test': True, 'report': False},
  'TRIAGE': {'test': False, 'report': False},
  'ROUTER': {'test': False, 'report': False},
  'GROUNDING': {'test': False, 'report': False},
  'SAVE_RESULTS': False,
  'PATH': './app/data/processed/reports/report_RAPIDO',
  
  'REFORMULATE': {'test': False, 'report': False}
}   

if file_list: 
  df = load_multiple_test_cases(file_list)
  df = validate_dataset_schema(df)

with open('./app/config/config.yaml', 'r') as file:
  config_data = yaml.load(file, Loader= yaml.FullLoader) 
  
client = RAGClient(config_data)
test_timestamps = {}

In [ ]:
import json

response_file_path = f'./app/data/processed/outcomes/{file_name}.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)
test_timestamps['general_tests'] = f'outcome_{file_name}.json'

In [ ]:
if test_config.get('GENERAL_TESTS'):
  results, reports = run_tests(
    config = test_config, 
    data = responses, 
    df = df, 
    timestamp = test_timestamps
)

In [ ]:
print(results)

In [35]:
import json
import pandas as pd

response_file_path = './app/data/processed/outcomes/outcome_20260429-100838.json'
with open(response_file_path, 'r', encoding='UTF-8') as f:
  responses = json.load(f)

In [50]:
counter = 0
columns = ['question', 'personality_answer', 'agent_answer', 'reason', 'context']
results = []
for item in responses:
  if item.get('partial_answers',{}).get('triage', {}).get('task', 'out_of_scope') == 'out_of_scope':
    continue
  
  if not item.get('partial_answers', {}).get('grounding', {}).get('is_grounded'):
    question = item.get('original_question')
    reason = item.get('partial_answers', {}).get('grounding', {}).get('reason', '')
    personality_answer = item.get('partial_answers', {}).get('personality', '')

    agent_answer = item.get('partial_answers', {}).get('agent_request', {}).get('data', {}).get('answer', '')
    context = item.get('partial_answers', {}).get('agent_request', {}).get('data', {}).get('context', '')

    results.append([question, personality_answer, agent_answer, reason, context])

In [51]:
df = pd.DataFrame(results, columns= columns)
df.to_excel('respuestas_descubrir.xlsx', index= False, sheet_name= 'descubrir')